<a href="https://colab.research.google.com/github/Fawada/AI-ML-course-notebooks/blob/main/module1/m2_notebook2_overfitting.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧪 Module 2 · Notebook 2
# Overfitting Exploration with Synthetic Data

---

**Module 2 · Deep Neural Networks**  
**Estimated time:** 40 minutes  
**Prerequisite:** Notebook 1 complete; Lesson 2 reading pages on Overfitting and Hyperparameters read

---

## 📋 What This Notebook Does

In Notebook 1 you trained a well-behaved MLP with dropout and early stopping. In this notebook you deliberately make things go wrong — and then fix them. This is the most effective way to build genuine intuition about overfitting, regularisation, and hyperparameter choices.

You will run four controlled experiments, each varying one thing at a time:

1. **Network depth** — what happens when you add more hidden layers?
2. **Learning rate** — what happens when it is too high or too low?
3. **Regularisation** — how do dropout, weight decay, and early stopping each reduce overfitting?
4. **Learning rate schedules** — does a decaying learning rate outperform a fixed one?

Finally, an **interactive widget** lets you combine all controls freely and explore the effects in real time.

---

## 🗺️ Notebook Structure

| Step | What you do |
|------|-------------|
| **0. Setup** | Install and import libraries |
| **1. Generate Data** | Create a synthetic binary classification dataset |
| **2. Experiment 1** | Vary network depth — observe train/val gap |
| **3. Experiment 2** | Vary learning rate — observe convergence |
| **4. Experiment 3** | Add regularisation — compare dropout, weight decay, early stopping |
| **5. Experiment 4** | Fixed vs scheduled learning rate |
| **6. Interactive** | Combine all controls with widgets |
| **7. Exercises** | Reflection and synthesis |

---

> **Why synthetic data?** Real datasets confound experiments — class imbalance, noisy features, and data quality all affect results. Synthetic data lets us control everything: we know the true data-generating process, we can set the difficulty precisely, and we can repeat experiments exactly.

---
## Step 0 — Setup

We will use `ipywidgets` for the interactive exploration section. Everything else is pre-installed on Google Colab.

In [ ]:
!pip install ipywidgets --quiet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import make_classification

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

import ipywidgets as widgets
from ipywidgets import interact, interactive_output, HBox, VBox, Label
from IPython.display import display

np.random.seed(42)
tf.random.set_seed(42)
sns.set(style='whitegrid')
%matplotlib inline

print(f'TensorFlow: {tf.__version__}')
print('✅ Setup complete.')

---
## Step 1 — Generate a Synthetic Dataset

We create a **binary classification** dataset with 2,000 samples and 20 features. We deliberately make it moderately difficult:

- `n_informative=10` — only 10 of the 20 features actually contain useful information; the other 10 are noise
- `n_redundant=5` — 5 features are linear combinations of the informative ones
- `class_sep=0.8` — the classes are not perfectly separable; there is genuine overlap

This combination ensures that a model must regularise properly to generalise — it is easy to overfit to the noisy features.

In [ ]:
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=10,
    n_redundant=5,
    n_repeated=0,
    class_sep=0.8,
    random_state=42
)

# Split: 60% train, 20% validation, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=42, stratify=y_train_full
)

# Standardise
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val   = scaler.transform(X_val)
X_test  = scaler.transform(X_test)

print(f'Training:   {X_train.shape[0]} samples × {X_train.shape[1]} features')
print(f'Validation: {X_val.shape[0]} samples')
print(f'Test:       {X_test.shape[0]} samples')
print(f'\nClass balance — train: {y_train.mean():.2f}, val: {y_val.mean():.2f}')
print('\nDataset ready. Each experiment below will use the same data split.')

---
## Step 2 — Experiment 1: Network Depth

### What are we testing?

We will train three networks with identical hyperparameters (same learning rate, no dropout, same epochs) but different depths:

- **Shallow:** 1 hidden layer (64 neurons)
- **Medium:** 3 hidden layers (128 → 64 → 32)
- **Deep:** 6 hidden layers (256 → 128 → 64 → 64 → 32 → 16)

### What to expect

Deeper networks have more parameters and more capacity to memorise. Without regularisation, adding depth tends to widen the train–validation gap — the hallmark of overfitting. Watch what happens to the gap as depth increases.

In [ ]:
def build_network(layer_sizes, dropout_rate=0.0, l2_lambda=0.0):
    """Build an MLP with configurable depth, dropout, and L2 regularisation."""
    model = keras.Sequential()
    model.add(keras.Input(shape=(20,)))
    for i, size in enumerate(layer_sizes):
        reg = regularizers.l2(l2_lambda) if l2_lambda > 0 else None
        model.add(layers.Dense(size, activation='relu',
                               kernel_regularizer=reg,
                               name=f'hidden_{i+1}'))
        if dropout_rate > 0:
            model.add(layers.Dropout(dropout_rate))
    model.add(layers.Dense(1, activation='sigmoid', name='output'))
    return model

def train_and_return(model, lr=0.001, epochs=100, verbose=0):
    """Compile, train, and return history."""
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    history = model.fit(
        X_train, y_train,
        epochs=epochs, batch_size=64,
        validation_data=(X_val, y_val),
        verbose=verbose
    )
    return history

# Train three depths
configs = {
    'Shallow (1 layer)':  [64],
    'Medium (3 layers)':  [128, 64, 32],
    'Deep (6 layers)':    [256, 128, 64, 64, 32, 16]
}

depth_histories = {}
print('Training networks of different depths (no regularisation)...')
for name, sizes in configs.items():
    m = build_network(sizes)
    h = train_and_return(m, lr=0.001, epochs=100)
    depth_histories[name] = h
    final_train = h.history['accuracy'][-1]
    final_val   = h.history['val_accuracy'][-1]
    print(f'  {name:25s}  train acc: {final_train:.3f}  val acc: {final_val:.3f}  gap: {final_train-final_val:.3f}')

print('\n✅ Done.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['steelblue', 'seagreen', 'tomato']

for ax, (name, h), color in zip(axes, depth_histories.items(), colors):
    ax.plot(h.history['accuracy'],     label='Train',      color=color,   linewidth=2)
    ax.plot(h.history['val_accuracy'], label='Validation', color=color,   linewidth=2, linestyle='--', alpha=0.7)
    gap = h.history['accuracy'][-1] - h.history['val_accuracy'][-1]
    ax.set_title(f'{name}\nFinal gap: {gap:.3f}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Accuracy', fontsize=11)
    ax.legend(fontsize=10)
    ax.set_ylim(0.4, 1.05)
    ax.grid(alpha=0.3)

plt.suptitle('Experiment 1: Effect of Network Depth on Overfitting (no regularisation)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 Observation: As depth increases, does the gap between training and validation accuracy grow?')
print('   This is the overfitting signature — the deeper network has memorised the training data.')

---
## Step 3 — Experiment 2: Learning Rate

### What are we testing?

We use the same 3-layer medium network but vary the learning rate across four values:

| Learning rate | Expected behaviour |
|---------------|--------------------------|
| 0.1 | Very high — likely to oscillate or diverge |
| 0.01 | High — may converge but unstably |
| 0.001 | Default Adam — typically reliable |
| 0.0001 | Very low — converges slowly, may underfit |

### What to expect

Too high: the loss curve will be jagged and may not decrease at all. Too low: the curve decreases smoothly but very slowly. The sweet spot produces a smooth, rapid decrease that plateaus cleanly.

In [ ]:
learning_rates = [0.1, 0.01, 0.001, 0.0001]
lr_histories   = {}

print('Training with different learning rates (same 3-layer architecture)...')
for lr in learning_rates:
    m = build_network([128, 64, 32])
    h = train_and_return(m, lr=lr, epochs=100)
    lr_histories[lr] = h
    final_val_loss = h.history['val_loss'][-1]
    print(f'  lr={lr:.4f}  final val loss: {final_val_loss:.4f}')

print('\n✅ Done.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colors = ['#E74C3C', '#E67E22', '#27AE60', '#2980B9']
labels = [f'lr = {lr}' for lr in learning_rates]

for h, color, label in zip(lr_histories.values(), colors, labels):
    axes[0].plot(h.history['loss'],     color=color, linewidth=2,   label=label)
    axes[1].plot(h.history['val_loss'], color=color, linewidth=2,   label=label)

for ax, title in zip(axes, ['Training Loss', 'Validation Loss']):
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Binary Cross-Entropy Loss', fontsize=11)
    ax.legend(fontsize=10)
    ax.set_ylim(0, 1.5)
    ax.grid(alpha=0.3)

plt.suptitle('Experiment 2: Effect of Learning Rate on Convergence',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 Observation: Which learning rate produces the smoothest, fastest convergence?')
print('   Which produces oscillation? Which is so slow it barely improves?')

---
## Step 4 — Experiment 3: Regularisation Techniques

### What are we testing?

We use the deliberately over-parameterised **deep (6-layer) network** from Experiment 1, which overfits clearly. We then apply three regularisation strategies — one at a time — and compare the results:

| Strategy | How it works |
|----------|--------------|
| **No regularisation (baseline)** | The overfitting case from Experiment 1 |
| **Dropout (0.4)** | Randomly zeroes 40% of neurons each step — forces redundancy |
| **L2 weight decay (λ=0.01)** | Adds a penalty to the loss for large weights — discourages complexity |
| **Early stopping** | Stops training when validation loss stops improving — limits overfitting duration |

> **Key question to keep in mind:** Does regularisation reduce the train-val gap by *lowering* training accuracy, *raising* validation accuracy, or both?

In [ ]:
deep_layers = [256, 128, 64, 64, 32, 16]
EPOCHS = 120

early_stop_cb = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=10, restore_best_weights=True
)

reg_configs = [
    ('No regularisation (baseline)', dict(dropout_rate=0.0, l2_lambda=0.0), []),
    ('Dropout (0.4)',                 dict(dropout_rate=0.4, l2_lambda=0.0), []),
    ('L2 weight decay (λ=0.01)',      dict(dropout_rate=0.0, l2_lambda=0.01), []),
    ('Early stopping',                dict(dropout_rate=0.0, l2_lambda=0.0), [early_stop_cb]),
]

reg_histories = {}
print('Training deep network with different regularisation strategies...')
for name, kwargs, callbacks in reg_configs:
    m = build_network(deep_layers, **kwargs)
    m.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy', metrics=['accuracy']
    )
    h = m.fit(X_train, y_train, epochs=EPOCHS, batch_size=64,
              validation_data=(X_val, y_val),
              callbacks=callbacks, verbose=0)
    reg_histories[name] = h
    tr_acc  = h.history['accuracy'][-1]
    val_acc = h.history['val_accuracy'][-1]
    ep      = len(h.history['loss'])
    print(f'  {name:35s}  epochs: {ep:3d}  train: {tr_acc:.3f}  val: {val_acc:.3f}  gap: {tr_acc-val_acc:.3f}')

print('\n✅ Done.')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.flat
colors = ['tomato', 'steelblue', 'seagreen', 'darkorchid']

for ax, (name, h), color in zip(axes, reg_histories.items(), colors):
    ax.plot(h.history['accuracy'],     label='Train',      color=color,   linewidth=2)
    ax.plot(h.history['val_accuracy'], label='Validation', color=color,   linewidth=2,
            linestyle='--', alpha=0.75)
    # Shade the gap
    tr  = np.array(h.history['accuracy'])
    val = np.array(h.history['val_accuracy'])
    epochs_arr = np.arange(1, len(tr)+1)
    ax.fill_between(epochs_arr, val, tr, alpha=0.12, color=color, label='Train-val gap')
    gap = tr[-1] - val[-1]
    ax.set_title(f'{name}\nFinal gap: {gap:.3f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel('Accuracy', fontsize=10)
    ax.legend(fontsize=9)
    ax.set_ylim(0.45, 1.05)
    ax.grid(alpha=0.3)

plt.suptitle('Experiment 3: Regularisation Strategies on the Deep (6-layer) Network\n'
             'Shaded region = train–validation gap (smaller = less overfitting)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 Which regularisation method closed the gap most effectively?')
print('   Did it raise validation accuracy, lower training accuracy, or both?')

---
## Step 5 — Experiment 4: Fixed vs Scheduled Learning Rate

### What are we testing?

A **fixed learning rate** keeps the step size constant throughout training. A **learning rate schedule** starts with a higher learning rate (for fast initial progress) and decays it over time (for fine-grained convergence at the end).

We will compare:
- **Fixed lr = 0.001** (the Adam default)
- **Exponential decay:** starts at 0.01, decays by factor 0.95 every 5 epochs
- **Cosine decay:** gradually decays from 0.01 to near zero following a cosine curve

The scheduled approach often converges faster and to a better final value — especially useful when training time is limited.

In [ ]:
EPOCHS_LR = 100
medium_layers = [128, 64, 32]

# Schedule 1: Exponential decay
lr_schedule_exp = keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=0.01,
    decay_steps=5 * (len(X_train) // 64),   # every 5 epochs
    decay_rate=0.95,
    staircase=True
)

# Schedule 2: Cosine decay
lr_schedule_cos = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.01,
    decay_steps=EPOCHS_LR * (len(X_train) // 64)
)

lr_exp_configs = [
    ('Fixed lr=0.001',      keras.optimizers.Adam(learning_rate=0.001)),
    ('Exponential decay',   keras.optimizers.Adam(learning_rate=lr_schedule_exp)),
    ('Cosine decay',        keras.optimizers.Adam(learning_rate=lr_schedule_cos)),
]

lr_sched_histories = {}
print('Comparing fixed vs scheduled learning rates...')
for name, optimizer in lr_exp_configs:
    m = build_network(medium_layers)
    m.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
    h = m.fit(X_train, y_train, epochs=EPOCHS_LR, batch_size=64,
              validation_data=(X_val, y_val), verbose=0)
    lr_sched_histories[name] = h
    print(f'  {name:25s}  final val acc: {h.history["val_accuracy"][-1]:.3f}')

print('\n✅ Done.')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
colors = ['tomato', 'steelblue', 'seagreen']

for (name, h), color in zip(lr_sched_histories.items(), colors):
    ax1.plot(h.history['loss'],         label=name, color=color, linewidth=2)
    ax2.plot(h.history['val_accuracy'], label=name, color=color, linewidth=2)

ax1.set_title('Training Loss', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(fontsize=10); ax1.grid(alpha=0.3)

ax2.set_title('Validation Accuracy', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.legend(fontsize=10); ax2.grid(alpha=0.3)

plt.suptitle('Experiment 4: Fixed vs Scheduled Learning Rate',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('📊 Which schedule converges fastest in the first 20 epochs?')
print('   Which achieves the highest final validation accuracy?')
print('   Does the cosine schedule show a smoother convergence than exponential?')

---
## Step 6 — Interactive Exploration: Combine All Controls

Now you can combine all the controls you have studied — depth, learning rate, dropout, and weight decay — and observe the combined effect. Use the sliders and dropdowns below to set up any configuration you like, then run the training and read the result.

> **Suggestion:** Start by reproducing the worst overfitting case you observed (deep + no regularisation + high LR). Then add regularisation one piece at a time and observe how each addition improves things.

In [ ]:
depth_widget    = widgets.Dropdown(options=[1, 2, 3, 4, 6], value=3, description='Depth:')
neurons_widget  = widgets.IntSlider(value=64, min=16, max=256, step=16, description='Neurons/layer:')
lr_widget       = widgets.SelectionSlider(
    options=[0.1, 0.01, 0.001, 0.0001], value=0.001, description='Learning rate:'
)
dropout_widget  = widgets.FloatSlider(value=0.0, min=0.0, max=0.6, step=0.1, description='Dropout:')
l2_widget       = widgets.FloatSlider(value=0.0, min=0.0, max=0.05, step=0.005, description='L2 lambda:',
                                      readout_format='.3f')
epochs_widget   = widgets.IntSlider(value=80, min=20, max=200, step=20, description='Epochs:')
earlystop_widget= widgets.Checkbox(value=False, description='Early stopping')
run_button      = widgets.Button(description='▶  Train Network', button_style='success',
                                 layout=widgets.Layout(width='180px', height='36px'))
output_area     = widgets.Output()

def on_run_clicked(b):
    with output_area:
        output_area.clear_output(wait=True)

        n_layers  = depth_widget.value
        n_neurons = neurons_widget.value
        lr        = lr_widget.value
        dr        = dropout_widget.value
        l2        = l2_widget.value
        n_epochs  = epochs_widget.value
        use_es    = earlystop_widget.value

        layer_sizes = [n_neurons] * n_layers
        m = build_network(layer_sizes, dropout_rate=dr, l2_lambda=l2)
        m.compile(
            optimizer=keras.optimizers.Adam(learning_rate=lr),
            loss='binary_crossentropy', metrics=['accuracy']
        )
        callbacks = []
        if use_es:
            callbacks.append(keras.callbacks.EarlyStopping(
                monitor='val_loss', patience=10, restore_best_weights=True
            ))

        print(f'Training: {n_layers} layers × {n_neurons} neurons | '
              f'lr={lr} | dropout={dr} | L2={l2:.3f} | '
              f'early_stop={use_es}...')

        h = m.fit(X_train, y_train, epochs=n_epochs, batch_size=64,
                  validation_data=(X_val, y_val), verbose=0)

        actual_epochs = len(h.history['loss'])
        final_train   = h.history['accuracy'][-1]
        final_val     = h.history['val_accuracy'][-1]
        gap           = final_train - final_val

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
        ax1.plot(h.history['loss'],         color='steelblue', linewidth=2, label='Train loss')
        ax1.plot(h.history['val_loss'],     color='tomato',    linewidth=2, linestyle='--', label='Val loss')
        ax1.set_title('Loss Curves'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
        ax1.legend(); ax1.grid(alpha=0.3)

        ax2.plot(h.history['accuracy'],     color='steelblue', linewidth=2, label='Train acc')
        ax2.plot(h.history['val_accuracy'], color='tomato',    linewidth=2, linestyle='--', label='Val acc')
        tr  = np.array(h.history['accuracy'])
        val = np.array(h.history['val_accuracy'])
        ax2.fill_between(range(actual_epochs), val, tr, alpha=0.15, color='grey', label='Gap')
        ax2.set_title('Accuracy Curves'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
        ax2.set_ylim(0.4, 1.05); ax2.legend(); ax2.grid(alpha=0.3)

        verdict = '✅ Well-regularised' if gap < 0.05 else ('⚠️  Mild overfitting' if gap < 0.12 else '❌ Significant overfitting')
        plt.suptitle(f'{verdict}   |   '
                     f'Train acc: {final_train:.3f}   Val acc: {final_val:.3f}   '
                     f'Gap: {gap:.3f}   Epochs run: {actual_epochs}',
                     fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()

run_button.on_click(on_run_clicked)

controls = VBox([
    HBox([depth_widget, neurons_widget]),
    HBox([lr_widget, epochs_widget]),
    HBox([dropout_widget, l2_widget]),
    HBox([earlystop_widget, run_button]),
])
display(controls, output_area)

---
## Step 7 — Exercises and Guided Reflection

Answer the questions below in your own words by replacing the placeholder text.

---

### ❓ Question 1 — Depth and Overfitting
**From Experiment 1: as you increased network depth (1 → 3 → 6 layers), what happened to the train–validation accuracy gap? Did all three networks achieve similar validation accuracy, or did deeper networks do better or worse?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 2 — Learning Rate
**From Experiment 2: which learning rate produced the most unstable (oscillating) training loss? Which was so small that training barely improved over 100 epochs? What would you choose as your default starting point and why?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 3 — Regularisation Comparison
**From Experiment 3: which regularisation method closed the overfitting gap most effectively? Did regularisation reduce training accuracy, raise validation accuracy, or both? Which technique would you reach for first in a real project and why?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 4 — Learning Rate Schedule
**From Experiment 4: did the scheduled learning rates converge to a better final validation accuracy than the fixed rate? Did one schedule converge faster in the early epochs? What advantage does scheduling give you when training time is limited?**

> **Your answer:** *(Replace this text with your observations)*

---

### ❓ Question 5 — Module Reflection Prompt
**The module asks: "Why might a shallow network outperform a deep one on small datasets?" Based on everything you have observed across both notebooks, write 2–3 sentences answering this question using specific evidence from your experiments.**

> **Your answer:** *(Replace this text with your observations)*

---
## ✅ Notebook 2 Complete — Module 2 Notebooks Finished

Across both notebooks in this module, you have produced:

| ✅ | Achievement |
|----|-------------|
| ✅ | A fully connected MLP in TensorFlow/Keras trained on SDSS stellar spectra with dropout and early stopping |
| ✅ | Training and validation loss and accuracy curves interpreted to assess convergence and generalisation |
| ✅ | A confusion matrix showing per-class classification performance across STAR, GALAXY, and QSO |
| ✅ | Visualisations of learned weight matrices from the first hidden layer, connecting what the network learned to the spectral input features |
| ✅ | A systematic overfitting exploration — varying depth, learning rate, regularisation, and learning rate schedule |
| ✅ | A comparison of fixed versus scheduled learning rates, observing the convergence difference in loss curves |
| ✅ | An interactive experiment combining all hyperparameter controls simultaneously |

---

### ➡️ What Comes Next

Module 3 builds directly on what you have learned here. In Module 3 you will study **autoencoders** — a class of neural network that uses the same MLP building blocks you have just mastered, but trains without any labels. The encoder-bottleneck-decoder structure you will encounter there is a direct extension of the layered MLP you built in Notebook 1.